# Admixture proportions from called genotypes (ADMIXTURE)

**Purpose.** Estimate admixture proportions the ordinary way, from genotypes that have
already been called, and then judge whether the answer can be trusted. Most of the
exercise is about that second part: choosing K, checking convergence across seeds, and
using evalAdmix to see where the model does not fit.

**What you will do**
 - run ADMIXTURE on an LD-pruned plink fileset and plot the proportions
 - see what LD does to the result if you do not prune
 - run several seeds at the same K and compare their likelihoods
 - use evalAdmix to find where the model fits badly, and compare a good and a bad K

**The data.** 192 individuals from the **1000 Genomes Project**: 16 populations, **12
individuals from each**, in four super-populations.

| Super-population | Populations |
|---|---|
| AFR (Africa) | ACB Barbados, ASW southwestern USA, ESN Esan Nigeria, GWD Gambia, MSL Mende Sierra Leone, YRI Yoruba Nigeria |
| AMR (Americas) | CLM Medellin Colombia, MXL Mexican ancestry Los Angeles, PEL Lima Peru |
| EAS (East Asia) | CHB Han Chinese Beijing, CHS Southern Han Chinese, JPT Japanese Tokyo |
| EUR (Europe) | CEU Utah northern/western European ancestry, GBR Britain, IBS Spain, TSI Italy |

**317,850 autosomal SNPs**, LD-pruned, as a plink binary fileset. The genotypes are
**called**, so ADMIXTURE can be used directly — unlike the low depth exercises where the
genotypes are never known.

**Before this** do [Admixture proportions from low depth sequencing](admixture_low_depth_human.ipynb).

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/admixture/human_called

# where you will do the exercise
WORK_DIR=$HOME/admixture_called_genotypes_human

mkdir -p $WORK_DIR/admix
echo $WORK_DIR > $HOME/.admixture_called_human_workdir
cd $WORK_DIR

# link the input files into the working folder
cp -sf $DATA/human_autosomes_12pp_pcaoneLD02* ./admix/ 2>/dev/null
cp -r -sf $DATA/precomputed ./admix/ 2>/dev/null

echo --programs that are installed:--
which admixture
which evalAdmix

echo; echo --- files in folder ---
ls admix/ | head

In [ ]:
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.admixture_called_human_workdir"))[1]
setwd(work_d)
getwd()

# Software and data

We will use PLINK, PCAone, ADMIXTURE, evalAdmix, and R plotting functions. First check that the programs are available.


In [ ]:
echo --programs that are installed:--
which admixture
which plink
which PCAone
which evalAdmix
which hapla



## Data sets

Make a working directory in your home folder and copy the starter files for the exercise. A lot of results throughout the exercise have been precomputed due to time constraints, but the commands used to generate them are shown.



## Metadata and population labels

The PLINK `.fam` file describes the individuals in the genotype data. The labels file maps individuals to population and broad ancestry region.


In [ ]:
echo -- number of individuals in fam file --
wc -l human_autosomes_12pp_pcaoneLD02.fam

echo -e "
-- first 10 lines of fam file --"
head human_autosomes_12pp_pcaoneLD02.fam

echo -e "
-- first 10 lines of label file: sample population region --"
head human_autosomes_12pp_pcaoneLD02.labels.tsv

echo -e "
-- population counts --"
awk '{print $2}' human_autosomes_12pp_pcaoneLD02.labels.tsv | sort | uniq -c

echo -e "
-- broad region counts --"
awk '{print $3}' human_autosomes_12pp_pcaoneLD02.labels.tsv | sort | uniq -c


**Questions**
 - How many individuals are in the file, and how many SNPs?
 - The design is balanced, 12 individuals per population. Why does that matter for an admixture analysis?


## The BIM file

The `.bim` file describes the genetic variants. This data set has already been pruned for LD with PCAone, so the variant count is much smaller than the original genome-wide BCF data.


In [ ]:
echo -- number of variants in PCAone-pruned data --
wc -l human_autosomes_12pp_pcaoneLD02.bim

echo -e "
-- first 10 variants --"
echo -e "CHR	variantID	CM	Position	allele_1	allele_2"
head human_autosomes_12pp_pcaoneLD02.bim

echo -e "
-- variants per chromosome --"
awk '{n[$1]++} END {for (c in n) print c, n[c]}' human_autosomes_12pp_pcaoneLD02.bim | sort -k1,1n



Run the code below to start a short quiz about the input data.


In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/called_quiz1.json")



## LD pruning with PCAone

ADMIXTURE assumes that markers are approximately independent. It is therefore common to prune variants in linkage disequilibrium before running ADMIXTURE.

Here we use PCAone for LD pruning because it estimates LD after accounting for population structure. This matters for data sets with individuals from multiple populations: ordinary LD pruning can confuse population structure with linkage disequilibrium.

The `-k` value in PCAone is the number of principal components used to model population structure before LD is estimated from residuals. It is not the same as ADMIXTURE's K, and it should not be interpreted as an expected number of ancestry components. Here we use `-k 10` as a conservative choice to capture several broad and within-region structure axes in the 1KGP subset. In practice you might want to be careful with this number, but plotting a PCA of your data, which you will learn about in the afternoon session, can help with this.

The current PCAone version uses a two-step workflow: first compute ancestry-adjusted residuals, then prune variants using those residuals. The commands are shown below, but the results are precomputed for this exercise.


In [ ]:
# These commands were used to generate the PCAone-pruned data.
# They are shown for reference and are not run in the notebook.
# In this command, -k is the number of PCs used for ancestry-adjusted LD, not ADMIXTURE K.

# PCAone -b human_autosomes_12pp_ids \
#   -k 10 \
#   -D \
#   --ld-stats 0 \
#   -n ${THREADS} \
#   -o pcaone_autosomes_k10

# PCAone -B pcaone_autosomes_k10.residuals \
#   --match-bim pcaone_autosomes_k10.mbim \
#   --ld-r2 0.2 \
#   --ld-bp 1000000 \
#   -n ${THREADS} \
#   -o pcaone_autosomes_k10_prune_ld02

# Use the precomputed PCAone pruning result for the rest of the exercise.
cp -a "${PRECOMP_RESULTS_DIR}/pcaone/." .

echo -- number of variants kept by PCAone pruning --
wc -l pcaone_autosomes_k10_prune_ld02.ld.prune.in

echo -e "
-- number of variants in extracted PLINK data --"
wc -l human_autosomes_12pp_pcaoneLD02.bim


### Quick check: LD pruning and ADMIXTURE setup


In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/called_quiz2_ld_admixture.json")



## ADMIXTURE

ADMIXTURE estimates each individual's ancestry proportions under a model with K ancestral populations. For this exercise we start with K=4 because we are working with data contains four broad regions: Africa (AFR), America (AMR), East Asia (EAS), and Europe (EUR).

The populations labelled AMR are admixed populations from the Americas. Therefore, we should not expect a simple one-to-one interpretation where each region becomes one pure component.

ADMIXTURE will return a model for the K you ask it to fit, even when that K is not a good description of the data. A clean-looking barplot is therefore not enough by itself: the solution may be unstable, underfit, or biologically misleading.


In [ ]:
admixture --help | head -40


**Question**
 - ADMIXTURE requires K as an argument. What does that mean for how you choose it?


ADMIXTURE starts from a random initial guess. Some seeds converge to a good solution and some can get stuck in a local optimum. We will first inspect a deliberately bad seed.

<img src="https://i.sstatic.net/GPErf.gif" alt="Animation of an optimizer moving across a bumpy likelihood landscape" width="520">

The above picture is only a metaphor, most likelihood landscapes will have many more dimensions, but the idea is useful: different random seeds can start the optimization in different parts of a complicated likelihood landscape, so they may end at different local optima. This is why we compare log likelihoods across many seeds instead of trusting a single ADMIXTURE run.


In [ ]:
# Command used to generate the bad seed result shown below.
# This is shown for reference and is not run in the notebook.

# admixture --seed ${BAD_SEED} -j ${THREADS} human_autosomes_12pp_pcaoneLD02.bed ${K}

# Load the precomputed bad-seed ADMIXTURE result.
cp -a "${PRECOMP_RESULTS_DIR}/bad_seed/." .

cat human_autosomes_12pp_pcaoneLD02.4.bad_seed2.log | tail -20


**Question**
 - This seed gave a poor result. Looking at the bar plot below, how would you recognise a bad run without being told?


### Plotting a "random" seed

The plot below shows ancestry proportions from seed 2. The populations are ordered roughly geographically: continental African populations, African diaspora populations, European populations, East Asian populations, and admixed American populations.


In [ ]:
library("repr")
options(repr.plot.width=17, repr.plot.height=5)
source("./visFuns.R")

labels <- read.table("human_autosomes_12pp_pcaoneLD02.labels.tsv", stringsAsFactors=FALSE)
pop <- labels[,2]
pop_order <- c("GWD", "MSL", "YRI", "ESN", "ACB", "ASW",
               "GBR", "CEU", "IBS", "TSI",
               "CHB", "CHS", "JPT",
               "MXL", "CLM", "PEL")
ord <- unlist(lapply(pop_order, function(p) which(pop == p)))

admix_cols <- c(AMR="#FF7F00", AFR="#4DAF4A", EUR="#377EB8", EAS="#984EA3")

reorder_components <- function(q) {
  ref <- c("PEL", "YRI", "GBR", "CHB")
  kord <- sapply(ref, function(p) which.max(colMeans(q[pop == p,,drop=FALSE])))
  if(length(unique(kord)) == ncol(q)) q[,kord] else q
}

plot_geo_admix <- function(q, title) {
  plotAdmix(q, pop=pop, ord=ord, rotatelab=45, padj=0.12,
            cex.lab=1.0, cex.main=1.3, main=title,
            colorpal=unname(admix_cols[c("AMR", "AFR", "EUR", "EAS")]))
}

q_bad <- reorder_components(as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.4.bad_seed2.Q")))
plot_geo_admix(q_bad, "ADMIXTURE proportions, K = 4, bad seed = 2")




- Does this result look plausible?
- Which populations or regions look strange?
- What could make ADMIXTURE produce a result like this?


## evalAdmix

evalAdmix diagnoses how well an ADMIXTURE model predicts the genotypes. It looks for correlations in residuals after fitting the model. Strong residual correlations tell us that individuals or populations are more alike (positive values) or different (negative values) than expected under the model. This in turn can indicate that the model is missing structure or that the ADMIXTURE run found a poor solution.

Run evalAdmix for the seed we just looked at. With four threads this should take a couple of minutes on the prepared data set, so it is normal if the cell sits for a little while before printing the final lines.


In [ ]:
# Run evalAdmix for the bad seed. If needed, the precomputed fallback is:
# cp -a "${PRECOMP_RESULTS_DIR}/evaladmix/human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval" .

/usr/bin/time -p evalAdmix \
  -plink human_autosomes_12pp_pcaoneLD02 \
  -fname human_autosomes_12pp_pcaoneLD02.4.bad_seed2.P \
  -qname human_autosomes_12pp_pcaoneLD02.4.bad_seed2.Q \
  -o human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval \
  -P ${THREADS} \
  > human_autosomes_12pp_pcaoneLD02.4.bad_seed2.evalAdmix.log 2>&1

tail -15 human_autosomes_12pp_pcaoneLD02.4.bad_seed2.evalAdmix.log
ls -lh human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval


In [ ]:
options(repr.plot.width=14, repr.plot.height=11)
r_bad <- as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval"))
plotCorRes(r_bad, pop=pop, ord=ord, max_z=0.05,
           rotatelabpop=20, adjlab=.05,
           title="evalAdmix residual correlations, bad seed = 2",
           cex.main=1.2, cex.lab=.9, cex.legend=.9)



- Which population pairs have strong residual correlations?
- Does evalAdmix agree with your visual impression from the ADMIXTURE plot?



## Checking convergence across seeds

To test whether ADMIXTURE has found a good optimum, we run the same K with multiple random seeds and compare log likelihoods. The runs below were precomputed with seeds 1 through 10.


**Questions**
 - Compare the bar plot with the population labels. Do the inferred clusters line up with the sampled populations?
 - Which individuals look admixed, and does that match where they were sampled?

In [ ]:
# Commands used to generate the multi-seed results.
# They are shown for reference and are not run in the notebook.

# mkdir -p multiRunK4
# for seed in 1 2 3 4 5 6 7 8 9 10
# do
#   admixture --seed $seed -j ${THREADS} human_autosomes_12pp_pcaoneLD02.bed ${K} \
#     > multiRunK4/human_autosomes_12pp_pcaoneLD02.4.log_$seed 2>&1
#   mv human_autosomes_12pp_pcaoneLD02.4.Q multiRunK4/human_autosomes_12pp_pcaoneLD02.4.Q_$seed
#   mv human_autosomes_12pp_pcaoneLD02.4.P multiRunK4/human_autosomes_12pp_pcaoneLD02.4.P_$seed
# done

# Load the precomputed likelihood summary from those runs.
mkdir -p multiRunK4
cp -a "${PRECOMP_RESULTS_DIR}/multiRunK4/." multiRunK4/

ls multiRunK4


**Question**
 - Several seeds were run at the same K. What varies between them, and which one should be kept?

In [ ]:
echo -- likelihoods sorted from best to worst --
cat multiRunK4/likelihoods_10seeds.tsv


### Quick check: seed convergence


In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/called_quiz3_convergence.json")



## Plotting the best seed

Seed 5 has the highest likelihood among the 10 runs. We now plot that solution and compare it to the bad seed.


In [ ]:
# Load the precomputed best-seed ADMIXTURE result.
cp -a "${PRECOMP_RESULTS_DIR}/best_seed/." .

cat human_autosomes_12pp_pcaoneLD02.4.best_seed5.log | tail -20


In [ ]:
options(repr.plot.width=17, repr.plot.height=5)
q_best <- reorder_components(as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q")))
plot_geo_admix(q_best, "ADMIXTURE proportions, K = 4, best seed = 5")



- How did the result change compared with seed 2?
- Which populations show clear evidence of admixture?
- Do the admixed American populations all have the same ancestry proportions?


## evalAdmix for the best seed

Finally, run evalAdmix for the best seed and compare the model fit.


In [ ]:
# Run evalAdmix for the best seed. If needed, the precomputed fallback is:
# cp -a "${PRECOMP_RESULTS_DIR}/evaladmix/human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval" .

/usr/bin/time -p evalAdmix \
  -plink human_autosomes_12pp_pcaoneLD02 \
  -fname human_autosomes_12pp_pcaoneLD02.4.best_seed5.P \
  -qname human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q \
  -o human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval \
  -P ${THREADS} \
  > human_autosomes_12pp_pcaoneLD02.4.best_seed5.evalAdmix.log 2>&1

tail -15 human_autosomes_12pp_pcaoneLD02.4.best_seed5.evalAdmix.log
ls -lh human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval


In [ ]:
options(repr.plot.width=14, repr.plot.height=11)
r_best <- as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval"))
plotCorRes(r_best, pop=pop, ord=ord, max_z=0.05,
           rotatelabpop=20, adjlab=.05,
           title="evalAdmix residual correlations, best seed = 5",
           cex.main=1.2, cex.lab=.9, cex.legend=.9)


## What if K is too low?

A bad random seed is one way to get a poor fit, but a model can also fit poorly because K is too small. Here we compare the K=4 result to a K=3 ADMIXTURE run. The K=3 run is precomputed, but the command below shows how it was generated.

ADMIXTURE will still return a model when K is too low; the problem is that the model may not be a good or biologically useful description. The likelihoods and evalAdmix residuals are checks on whether the solution is stable enough to interpret.


**Questions**
 - Two runs at the same K reached different likelihoods. What does that tell you about the likelihood surface?
 - How many seeds would you run before trusting a result?

In [ ]:
# Command used to generate the K=3 result shown below.
# This is shown for reference and is not run in the notebook.

# admixture --seed ${BEST_SEED} -j ${THREADS} human_autosomes_12pp_pcaoneLD02.bed 3

# Load the precomputed K=3 ADMIXTURE result.
cp -a "${PRECOMP_RESULTS_DIR}/k3_underfit/." .

cat human_autosomes_12pp_pcaoneLD02.3.seed5.log | tail -20


In [ ]:
options(repr.plot.width=17, repr.plot.height=5)
q_k3 <- as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.3.seed5.Q"))
ref_k3 <- c("GBR", "CHB", "YRI")
kord_k3 <- sapply(ref_k3, function(p) which.max(colMeans(q_k3[pop == p,,drop=FALSE])))
if(length(unique(kord_k3)) == ncol(q_k3)) q_k3 <- q_k3[,kord_k3]
plotAdmix(q_k3, pop=pop, ord=ord, rotatelab=45, padj=0.12,
          cex.lab=1.0, cex.main=1.3,
          main="ADMIXTURE proportions, K = 3",
          colorpal=unname(admix_cols[c("EUR", "EAS", "AFR")]))



- Which ancestry patterns are forced together when K=3?
- Does K=3 make biological sense for these populations?
- What kinds of structure would you expect evalAdmix to flag for this model?


In [ ]:
# Optional: uncomment this block if you want to run evalAdmix for the K=3 model yourself.

# /usr/bin/time -p evalAdmix \
#   -plink human_autosomes_12pp_pcaoneLD02 \
#   -fname human_autosomes_12pp_pcaoneLD02.3.seed5.P \
#   -qname human_autosomes_12pp_pcaoneLD02.3.seed5.Q \
#   -o human_autosomes_12pp_pcaoneLD02.3.seed5.eval \
#   -P ${THREADS} \
#   > human_autosomes_12pp_pcaoneLD02.3.seed5.evalAdmix.log 2>&1

# Default: copy the precomputed K=3 evalAdmix result.
cp -a "${PRECOMP_RESULTS_DIR}/evaladmix/human_autosomes_12pp_pcaoneLD02.3.seed5.eval" .

# The log is only used to show what a completed run looks like.
cp -a "${PRECOMP_RESULTS_DIR}/k3_underfit/human_autosomes_12pp_pcaoneLD02.3.seed5.evalAdmix.log" . 2>/dev/null || true

if [ -f human_autosomes_12pp_pcaoneLD02.3.seed5.evalAdmix.log ]; then
  tail -15 human_autosomes_12pp_pcaoneLD02.3.seed5.evalAdmix.log
fi
ls -lh human_autosomes_12pp_pcaoneLD02.3.seed5.eval


In [ ]:
options(repr.plot.width=14, repr.plot.height=11)
r_k3 <- as.matrix(read.table("human_autosomes_12pp_pcaoneLD02.3.seed5.eval"))
plotCorRes(r_k3, pop=pop, ord=ord, max_z=0.05,
           rotatelabpop=20, adjlab=.05,
           title="evalAdmix residual correlations, K = 3",
           cex.main=1.2, cex.lab=.9, cex.legend=.9)


- How does the K=3 residual plot differ from the K=4 best-seed residual plot?


In [ ]:
summarize_eval <- function(file) {
  m <- as.matrix(read.table(file))
  m[upper.tri(m, diag=TRUE)] <- NA
  c(mean_abs=mean(abs(m), na.rm=TRUE),
    p95_abs=unname(quantile(abs(m), .95, na.rm=TRUE)),
    max_abs=max(abs(m), na.rm=TRUE))
}

rbind(
  k3_seed5=summarize_eval("human_autosomes_12pp_pcaoneLD02.3.seed5.eval"),
  bad_seed2=summarize_eval("human_autosomes_12pp_pcaoneLD02.4.bad_seed2.eval"),
  best_seed5=summarize_eval("human_autosomes_12pp_pcaoneLD02.4.best_seed5.eval")
)


### Quick check: evalAdmix model fit


In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/called_quiz4_evaladmix.json")


## Comparing several K values

K=4 is still the main result in this exercise because it has a simple interpretation for these populations and we checked convergence across seeds. It is still useful to see what happens when K is too low or when K is increased beyond the main model.

The K=2 and K=3 examples show underfitting: several ancestry patterns are forced together. The K=5, K=6, and K=7 examples below are exploratory precomputed runs stopped after 15 main iterations to keep this comparison practical. They are not meant as final, converged model choices. They show a different danger: as K increases, ADMIXTURE can create increasingly fine-scale components that are tempting to label, even when they may reflect sampling, local optima, or structure that is not relevant to the question being asked.


In [ ]:

cd "${WORK_DIR}"

cp -a "${PRECOMP_RESULTS_DIR}/k_ladder/." .

echo -- K ladder files --
ls -lh \
  human_autosomes_12pp_pcaoneLD02.2.seed5.Q \
  human_autosomes_12pp_pcaoneLD02.3.seed5.Q \
  human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q \
  human_autosomes_12pp_pcaoneLD02.5.seed5_iter15.Q \
  human_autosomes_12pp_pcaoneLD02.6.seed5_iter15.Q \
  human_autosomes_12pp_pcaoneLD02.7.seed5_iter15.Q

echo
echo -- K ladder run notes --
cat k_ladder_summary.tsv


In [ ]:
# all of this is just plotting code to make the figure look a little nicer,
# dont worry about it if you dont understand it

options(repr.plot.width=15, repr.plot.height=11)
source("./visFuns.R")

labels_k <- read.table("human_autosomes_12pp_pcaoneLD02.labels.tsv",
                       stringsAsFactors=FALSE)
pop_k <- labels_k[, 2]
pop_order_k <- c("GWD", "MSL", "YRI", "ESN", "ACB", "ASW",
                 "GBR", "CEU", "IBS", "TSI",
                 "CHB", "CHS", "JPT",
                 "MXL", "CLM", "PEL")
ord_k <- orderInds(pop=pop_k, popord=pop_order_k)

q_files_k <- c(
  "2"="human_autosomes_12pp_pcaoneLD02.2.seed5.Q",
  "3"="human_autosomes_12pp_pcaoneLD02.3.seed5.Q",
  "4"="human_autosomes_12pp_pcaoneLD02.4.best_seed5.Q",
  "5"="human_autosomes_12pp_pcaoneLD02.5.seed5_iter15.Q",
  "6"="human_autosomes_12pp_pcaoneLD02.6.seed5_iter15.Q",
  "7"="human_autosomes_12pp_pcaoneLD02.7.seed5_iter15.Q"
)

admix_cols_k <- c(AMR="#FF7F00", AFR="#4DAF4A", EUR="#377EB8", EAS="#984EA3")
extra_cols_k <- c("#A65628", "#F781BF", "#999999", "#E41A1C")

pick_components_k <- function(q, anchors, anchor_cols) {
  used <- integer(0)
  cols <- character(0)
  for (i in seq_along(anchors)) {
    anchor <- anchors[i]
    means <- colMeans(q[pop_k == anchor, , drop=FALSE])
    means[used] <- -Inf
    k <- which.max(means)
    if (is.finite(means[k]) && !(k %in% used)) {
      used <- c(used, k)
      cols <- c(cols, anchor_cols[i])
    }
    if (length(used) == ncol(q)) break
  }
  extra <- setdiff(seq_len(ncol(q)), used)
  list(order=c(used, extra), colors=c(cols, extra_cols_k[seq_along(extra)]))
}

component_setup_k <- function(q, k_label) {
  if (k_label == "2") {
    return(pick_components_k(q, anchors=c("YRI"),
                             anchor_cols=unname(admix_cols_k["AFR"])))
  }
  if (k_label == "3") {
    return(pick_components_k(q, anchors=c("YRI", "GBR", "CHB"),
                             anchor_cols=unname(admix_cols_k[c("AFR", "EUR", "EAS")])))
  }
  pick_components_k(q, anchors=c("PEL", "YRI", "GBR", "CHB", "ACB", "MXL", "IBS", "CLM"),
                    anchor_cols=c(unname(admix_cols_k[c("AMR", "AFR", "EUR", "EAS")]), extra_cols_k))
}

layout(matrix(seq_along(q_files_k), ncol=1))
for (i in seq_along(q_files_k)) {
  k_label <- names(q_files_k)[i]
  q <- as.matrix(read.table(q_files_k[i]))
  setup_k <- component_setup_k(q, k_label)
  q <- q[, setup_k$order, drop=FALSE]

  par(mar=c(ifelse(i == length(q_files_k), 5.5, 1.1), 4.6, 2.0, 1),
      xpd=FALSE)
  h <- barplot(t(q)[, ord_k], col=setup_k$colors,
               space=0, border=NA, axes=FALSE,
               ylab=ifelse(i == ceiling(length(q_files_k) / 2),
                           "Admixture proportions", ""),
               main=paste0("K = ", k_label),
               cex.main=1.1, cex.lab=1.0)
  axis(2, at=c(0, 0.5, 1), las=1, cex.axis=0.75)
  abline(v=cumsum(as.numeric(table(factor(pop_k[ord_k], levels=unique(pop_k[ord_k]))))),
         col="black", lwd=0.7)

  if (i == length(q_files_k)) {
    pop_centers <- tapply(h, factor(pop_k[ord_k], levels=unique(pop_k[ord_k])), mean)
    text(pop_centers, -0.08, names(pop_centers), srt=45, adj=1,
         cex=0.7, xpd=NA)
  }
}
layout(1)



- Which population structure is merged when K is too low?
- Which extra components at K=5 to K=7 look like plausible fine-scale structure, and which look harder to defend?
- Why does the likelihood generally improve as K increases, and why is that not enough to choose the largest K?
- What additional checks would you want before interpreting one of the higher-K models?
